In [ ]:
# All the imports Here
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [ ]:
#  Dataset Loading
df=pd.read_csv("german_credit_data1.csv")
df.head(5)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['Risk'].describe()

In [ ]:
# Missing values Checking
df.isnull().sum()

In [ ]:
# Fill the Missing values 
df['Checking account'].fillna('Unknown',inplace=True)
df['Saving accounts'].fillna('Unknown',inplace=True)

In [ ]:
#Cleaned
df.isnull().sum()

In [ ]:
plt.scatter(df['Credit amount'], df['Duration'])

In [ ]:
colors = df['Risk'].map({'good':'green','bad':'red'})
plt.scatter(df['Credit amount'], df['Duration'], c=colors)
plt.xlabel("Credit Amount")
plt.ylabel("Duration")
plt.show()

In [ ]:
sns.countplot(x='Risk', data=df)
plt.title("Distribution of Loan Risk")
plt.show()

In [ ]:
#Target and features
x=df.drop('Risk',axis=1)
y=df['Risk']

In [ ]:
# Devide the numerical and categorical columns
num_cols = ['Age', 'Job', 'Credit amount', 'Duration']
cat_cols = ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
    ]
)

In [ ]:
x.shape

In [ ]:

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

model=Pipeline(steps=[
    ("Preprocessing",preprocessor),
    ("Classifier",LogisticRegression(max_iter=1000,class_weight='balanced') )
])

model.fit(x_train, y_train)

In [ ]:
model.score(x_test,y_test)

In [ ]:
y_pred=model.predict(x_test)
print(classification_report(y_test,y_pred))

In [ ]:
new_customer=pd.DataFrame({
    'Age':[30],
    'Sex':['male'],
    'Job':[2],
    'Housing':['own'],
    'Saving accounts':['moderate'],
    'Checking account':['moderate'],
    'Credit amount':[5000],
    'Duration':[12],
    'Purpose':['car']
})

risk_pred=model.predict(new_customer)
risk=model.predict_proba(new_customer)[0][1]

print("predicted risk:",risk_pred[0])
print("probablity of being good risk:",risk)

In [ ]:
prob_good = model.predict_proba(new_customer)[0][1]

if prob_good >= 0.60:
    decision = "Loan Approved - Low Risk 🟢"

elif 0.50 <= prob_good < 0.60:
    decision = "Manual Review Required 🟡"

else:
    decision = "Loan Rejected - High Risk 🔴"

print("Probability:", prob_good)
print("Final Decision:", decision)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = model.predict(x_test)

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

y_prob = model.predict_proba(x_test)[:,1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob, pos_label="good")
auc_score = roc_auc_score(y_test, y_prob)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (AUC = {auc_score:.2f})")
plt.show()

In [ ]:
import numpy as np

feature_names = model.named_steps["Preprocessing"].get_feature_names_out()
coefficients = model.named_steps["Classifier"].coef_[0]

importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

importance = importance.sort_values(by="Coefficient", ascending=False)

print(importance.head(10))

In [ ]:
y_prob = model.predict_proba(x_test)[:,1]
plt.hist(y_prob)
plt.show()